In [20]:
import numpy as np
import pandas as pd
import time
from scipy.spatial.distance import cdist
from sklearn.datasets import (load_iris,load_wine,load_breast_cancer,load_diabetes,load_digits)
from sklearn.preprocessing import StandardScaler

In [21]:
def greedy_kcenter(D, k):
  n = len(D)
  centres = [0]
  nearest = D[:, 0].copy()

  while len(centres) < k:
      next_centre = np.argmax(nearest)
      centres.append(next_centre)
      nearest = np.minimum(nearest, D[:, next_centre])

  cost = np.max(nearest)
  return centres, cost

In [22]:
def local_search_kcenter(D, k, max_iter=30):
  n = len(D)

  rng = np.random.default_rng(42)
  centres = list(rng.choice(n, k, replace=False))

  for iteration in range(max_iter):
    distances = D[:, centres]

    order = np.argsort(distances, axis=1)

    nearest = distances[np.arange(n), order[:, 0]]
    second = distances[np.arange(n), order[:, 1]]

    nearest_pos = order[:, 0]

    best_cost = np.max(nearest)
    best_swap = None

    non_centres = [i for i in range(n) if i not in centres]

    for pos in range(k):
      base = np.where(nearest_pos == pos, second, nearest)
      for candidate in non_centres:
        new_distances = np.minimum(base, D[:, candidate])
        new_cost = np.max(new_distances)
        if new_cost < best_cost:
          best_cost = new_cost
          best_swap = (pos, candidate)

    if best_swap is None:
      break

    centres[best_swap[0]] = best_swap[1]

  final_cost = np.max(np.min(D[:, centres], axis=1))

  return centres, final_cost

In [23]:
print(" TOY DATASET")

points = np.array([
    [0.0, 0.0],
    [1.0, 1.2],
    [2.3, 0.2],
    [3.5, 1.3],
    [4.2, 3.0],
    [1.2, 3.2],
    [2.6, 2.2]
])

labels = ["A", "B", "C", "D", "E", "F", "G"]
k = 3

 TOY DATASET


In [24]:
# Euclidean distance matrix
D = cdist(points, points, metric="euclidean")

#  Greedy 2-Approximation
start = time.perf_counter()
g_centres, g_cost = greedy_kcenter(D, k)
g_time = time.perf_counter() - start

# Local Search Heuristic
start = time.perf_counter()
h_centres, h_cost = local_search_kcenter(D, k)
h_time = time.perf_counter() - start

print("\nGreedy 2-Approximation")
print("Centres:", [labels[i] for i in g_centres])
print("Cost:", round(g_cost, 3))
print("Time:", round(g_time, 6), "seconds")

print("\nLocal Search Heuristic")
print("Centres:", [labels[i] for i in h_centres])
print("Cost:", round(h_cost, 3))
print("Time:", round(h_time, 6), "seconds")


Greedy 2-Approximation
Centres: ['A', 'E', 'F']
Cost: 2.309
Time: 0.000183 seconds

Local Search Heuristic
Centres: ['G', 'B', 'E']
Cost: 1.72
Time: 0.000563 seconds


In [25]:
datasets = {
    "Iris": load_iris().data,
    "Wine": load_wine().data,
    "Breast Cancer": load_breast_cancer().data,
    "Diabetes": load_diabetes().data,
    "Digits": load_digits().data}

k = 5

greedy_results = []
heuristic_results = []

In [26]:
print("GREEDY 2-APPROXIMATION :")
greedy_results = []
for name, data in datasets.items():
  print("\nProcessing:", name)

  X = StandardScaler().fit_transform(data)

  #distance matrix
  D = cdist(X, X, metric="euclidean")

  #execution time
  start = time.perf_counter()
  centres, cost = greedy_kcenter(D, k)
  execution_time = time.perf_counter() - start

  greedy_results.append({"Dataset": name,"Observations": len(X),"Features": X.shape[1],"Greedy Centres": centres,"Greedy Cost": cost,"Greedy Time": execution_time})

  print("Centres:", centres)
  print("Cost:", round(cost, 4))
  print("Time:", round(execution_time, 6), "seconds")
print()
greedy_df = pd.DataFrame(greedy_results)

print("\nGREEDY 2-APPROXIMATION RESULTS")
display(greedy_df)


GREEDY 2-APPROXIMATION :

Processing: Iris
Centres: [0, np.int64(118), np.int64(60), np.int64(70), np.int64(131)]
Cost: 2.2121
Time: 6.7e-05 seconds

Processing: Wine
Centres: [0, np.int64(146), np.int64(115), np.int64(121), np.int64(59)]
Cost: 6.6696
Time: 6.2e-05 seconds

Processing: Breast Cancer
Centres: [0, np.int64(152), np.int64(561), np.int64(212), np.int64(213)]
Cost: 12.3181
Time: 5.5e-05 seconds

Processing: Diabetes
Centres: [0, np.int64(123), np.int64(441), np.int64(187), np.int64(117)]
Cost: 5.0589
Time: 6.5e-05 seconds

Processing: Digits
Centres: [0, np.int64(988), np.int64(502), np.int64(1264), np.int64(1070)]
Cost: 31.8883
Time: 0.0002 seconds


GREEDY 2-APPROXIMATION RESULTS


,Dataset,Observations,Features,Greedy Centres,Greedy Cost,Greedy Time
0,Iris,150,4,"[0, 118, 60, 70, 131]",2.212083,0.000067
1,Wine,178,13,"[0, 146, 115, 121, 59]",6.669569,0.000062
2,Breast Cancer,569,30,"[0, 152, 561, 212, 213]",12.318072,0.000055
3,Diabetes,442,10,"[0, 123, 441, 187, 117]",5.058893,0.000065
4,Digits,1797,64,"[0, 988, 502, 1264, 1070]",31.888310,0.000200


In [27]:
print(" LOCAL SEARCH HEURISTIC :")
heuristic_results = []
for name, data in datasets.items():
  print()
  print("Processing:", name)

  X = StandardScaler().fit_transform(data)

  # distance matrix
  D = cdist(X, X, metric="euclidean")

  #execution time
  start = time.perf_counter()
  centres, cost = local_search_kcenter(D, k)
  execution_time = time.perf_counter() - start

  heuristic_results.append({"Dataset": name,"Heuristic Centres": centres,"Heuristic Cost": cost,"Heuristic Time": execution_time})

  print("Centres:", centres)
  print("Cost:", round(cost, 4))
  print("Time:", round(execution_time, 6), "seconds")

print()
heuristic_df = pd.DataFrame(heuristic_results)

print("LOCAL SEARCH HEURISTIC RESULTS")
display(heuristic_df)

 LOCAL SEARCH HEURISTIC :

Processing: Iris
Centres: [5, 125, 119, np.int64(13), np.int64(64)]
Cost: 1.7232
Time: 0.04888 seconds

Processing: Wine
Centres: [43, 100, 20, 4, np.int64(177)]
Cost: 4.7052
Time: 0.03847 seconds

Processing: Breast Cancer
Centres: [212, 68, 0, 62, 13]
Cost: 10.669
Time: 0.239776 seconds

Processing: Diabetes
Centres: [0, 134, 12, 324, 168]
Cost: 4.1506
Time: 0.182398 seconds

Processing: Digits
Centres: [1264, 988, 502, 1053, 87]
Cost: 30.2029
Time: 0.662451 seconds

LOCAL SEARCH HEURISTIC RESULTS


,Dataset,Heuristic Centres,Heuristic Cost,Heuristic Time
0,Iris,"[5, 125, 119, 13, 64]",1.723237,0.048880
1,Wine,"[43, 100, 20, 4, 177]",4.705204,0.038470
2,Breast Cancer,"[212, 68, 0, 62, 13]",10.669004,0.239776
3,Diabetes,"[0, 134, 12, 324, 168]",4.150593,0.182398
4,Digits,"[1264, 988, 502, 1053, 87]",30.202894,0.662451


In [28]:
greedy_df = pd.DataFrame(greedy_results)
heuristic_df = pd.DataFrame(heuristic_results)

final_df = pd.merge(greedy_df,heuristic_df,on="Dataset")

print("FINAL COMPARISON :")
display(final_df.round(6))

FINAL COMPARISON :


,Dataset,Observations,Features,Greedy Centres,Greedy Cost,Greedy Time,Heuristic Centres,Heuristic Cost,Heuristic Time
0,Iris,150,4,"[0, 118, 60, 70, 131]",2.212083,0.000067,"[5, 125, 119, 13, 64]",1.723237,0.048880
1,Wine,178,13,"[0, 146, 115, 121, 59]",6.669569,0.000062,"[43, 100, 20, 4, 177]",4.705204,0.038470
2,Breast Cancer,569,30,"[0, 152, 561, 212, 213]",12.318072,0.000055,"[212, 68, 0, 62, 13]",10.669004,0.239776
3,Diabetes,442,10,"[0, 123, 441, 187, 117]",5.058893,0.000065,"[0, 134, 12, 324, 168]",4.150593,0.182398
4,Digits,1797,64,"[0, 988, 502, 1264, 1070]",31.888310,0.000200,"[1264, 988, 502, 1053, 87]",30.202894,0.662451


In [29]:
final_df.to_csv("kcenter_comparison.csv", index=False)
print("\nResults saved successfully!")


Results saved successfully!
